# God's Eye · application de détection, **version 2**

Copie de `detection-app.ipynb`, avec deux évolutions :
1. **Cellule 4 (Alerte sinistre)** : chaque classe a sa couleur, y compris celles qui n'apparaissent qu'à T1 ; chaque cadre porte son étiquette ; la légende affiche l'inventaire **T0 → T1** (ex. `bridge : 2 -> 0`, `harbor : 0 -> 5`).
2. **Nouvelle cellule 4 bis (Analyse des transformations)** : appariement spatial des objets par recouvrement, avec un verdict par objet (disparu, transformation plausible, incohérence du modèle) et une confirmation possible par une source indépendante (Spirit Digger).

Les autres cellules sont inchangées. Le notebook d'origine n'a pas été modifié.


In [ ]:
!pip install -q ultralytics


In [ ]:
pip install sahi

In [ ]:
# ==========================================
# CELLULE 1 : IMPORTATIONS & INITIALISATION
# ==========================================
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TABLEAU_COLORS
from PIL import Image
from collections import Counter

from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# --- PARAMÈTRES DU SYSTÈME ---
MODEL_PATH = "/kaggle/input/datasets/ayiamavigan/xview-dota-ckpt-backup/run_state/weights/best.pt"
CONFIDENCE_THRESHOLD = 0.25
DEVICE = "cuda:0" # Passer à "cpu" si besoin

# Couleurs pour notre légende (palette esthétique)
COLORS = list(TABLEAU_COLORS.values()) 

print("Chargement du modèle de surveillance satellitaire en cours...")
try:
    detection_model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model_path=MODEL_PATH,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        device=DEVICE,
    )
    print("Modèle chargé et sécurisé. Prêt pour l'analyse.")
except Exception as e:
    print(f"Erreur lors du chargement du modèle : {e}")

In [ ]:
# ==========================================
# CELLULE 2 : MOTEUR D'ANALYSE & RENDU VISUEL
# ==========================================

def get_chronological_images(folder_path):
    """
    Récupère et trie les images d'un dossier chronologiquement 
    (basé sur le nom du fichier, ex: T0.tif, T1.tif...).
    """
    valid_extensions = ('.tif', '.tiff', '.jpg', '.jpeg', '.png')
    images = [os.path.join(folder_path, f) for f in os.listdir(folder_path) 
              if f.lower().endswith(valid_extensions)]
    # Tri alphabétique qui simule la chronologie
    images.sort()
    return images

def scan_image(image_path):
    """
    Exécute SAHI sur l'image et retourne les prédictions.
    """
    result = get_sliced_prediction(
        image_path,
        detection_model,
        slice_height=640,
        slice_width=640,
        overlap_height_ratio=0.20,
        overlap_width_ratio=0.20,
    )
    return result.object_prediction_list

def display_strategic_map(image_path, predictions, title="Rapport d'Analyse Satellitaire"):
    """
    Affiche l'image avec les cadres de détection, et place une légende colorée
    à droite de l'image pour éviter de la surcharger.
    """
    # 1. Préparation de l'image et des statistiques
    image_np = np.array(Image.open(image_path).convert("RGB"))
    labels_present = [pred.category.name for pred in predictions]
    counts = Counter(labels_present)
    
    # Création d'un dictionnaire de couleurs dynamique pour les classes trouvées
    unique_classes = list(counts.keys())
    class_colors = {cls_name: COLORS[i % len(COLORS)] for i, cls_name in enumerate(unique_classes)}
    
    # 2. Configuration de la figure Matplotlib
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.imshow(image_np)
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    ax.axis("off")
    
    # 3. Dessin des boîtes (cadres) sur l'image
    for pred in predictions:
        bbox = pred.bbox
        cls_name = pred.category.name
        color = class_colors[cls_name]
        
        # Création du rectangle (sans remplissage)
        rect = mpatches.Rectangle(
            (bbox.minx, bbox.miny), 
            bbox.maxx - bbox.minx, 
            bbox.maxy - bbox.miny,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        
    # 4. Création de la légende déportée sur le côté droit
    legend_patches = []
    for cls_name, count in counts.items():
        color = class_colors[cls_name]
        # On ajoute le nombre d'objets dans la légende
        label_text = f"{cls_name} ({count})"
        patch = mpatches.Patch(color=color, label=label_text)
        legend_patches.append(patch)
        
    if legend_patches:
        # bbox_to_anchor permet de sortir la légende de l'image (espace blanc)
        ax.legend(
            handles=legend_patches, 
            loc='center left', 
            bbox_to_anchor=(1.02, 0.5), 
            title="Inventaire Détecté",
            title_fontsize='14',
            fontsize='12',
            frameon=True,
            shadow=True
        )
    else:
        # Si rien n'est détecté
        fig.text(0.95, 0.5, "AUCUNE DÉTECTION", fontsize=14, color='red', va='center')
        
    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================================
# CELLULE 3 : BRIQUE 1 - INVENTAIRE MULTIPLE ET SAUVEGARDE
# ==========================================
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from collections import Counter

# 1. Creation du dossier de sortie pour cette mission
OUTPUT_DIR = "Inventaire_Detection_Resultats"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[DOSSIER CREE] Les resultats seront sauvegardes dans le repertoire : {OUTPUT_DIR}")

# 2. Redefinition de la fonction d'affichage pour integrer la sauvegarde
def display_and_save_strategic_map(image_path, predictions, title, output_dir):
    image_np = np.array(Image.open(image_path).convert("RGB"))
    labels_present = [pred.category.name for pred in predictions]
    counts = Counter(labels_present)
    
    unique_classes = list(counts.keys())
    # Utilisation de la variable COLORS definie dans la Cellule 1
    class_colors = {cls_name: COLORS[i % len(COLORS)] for i, cls_name in enumerate(unique_classes)}
    
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.imshow(image_np)
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    ax.axis("off")
    
    for pred in predictions:
        bbox = pred.bbox
        cls_name = pred.category.name
        color = class_colors[cls_name]
        
        rect = mpatches.Rectangle(
            (bbox.minx, bbox.miny), 
            bbox.maxx - bbox.minx, 
            bbox.maxy - bbox.miny,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        
    legend_patches = []
    for cls_name, count in counts.items():
        color = class_colors[cls_name]
        label_text = f"{cls_name} ({count})"
        patch = mpatches.Patch(color=color, label=label_text)
        legend_patches.append(patch)
        
    if legend_patches:
        ax.legend(
            handles=legend_patches, 
            loc='center left', 
            bbox_to_anchor=(1.02, 0.5), 
            title="Inventaire Detecte",
            title_fontsize='14',
            fontsize='12',
            frameon=True,
            shadow=True
        )
    else:
        fig.text(0.95, 0.5, "AUCUNE DETECTION", fontsize=14, color='red', va='center')
        
    plt.tight_layout()
    
    # SAUVEGARDE DE LA FIGURE AVANT AFFICHAGE
    nom_fichier = f"analyse_{os.path.basename(image_path)}"
    chemin_sauvegarde = os.path.join(output_dir, nom_fichier)
    # bbox_inches='tight' est crucial pour ne pas couper la legende qui est a l'exterieur
    plt.savefig(chemin_sauvegarde, bbox_inches='tight', dpi=300)
    print(f"[SAUVEGARDE] Rapport visuel enregistre sous : {chemin_sauvegarde}")
    
    plt.show()

# 3. Lancement du processus
print("\n[SYSTEME DE SURVEILLANCE - MODE INVENTAIRE DE MASSE]")
# Demande de fournir les chemins separes par des virgules
chemins_input = input("Entrez les chemins absolus des images (separes par une virgule) : ")

# Nettoyage de la saisie utilisateur pour creer une liste propre
liste_chemins = [chemin.strip() for chemin in chemins_input.split(",") if chemin.strip()]

if not liste_chemins:
    print("[ERREUR] Aucune image n'a ete fournie.")
else:
    print(f"\n[DEMARRAGE] {len(liste_chemins)} image(s) a analyser.\n")
    
    for image_path in liste_chemins:
        if not os.path.isfile(image_path):
            print(f"[IGNORER] Le fichier n'existe pas : {image_path}")
            continue
            
        nom_image = os.path.basename(image_path)
        print("-" * 60)
        print(f"[SCAN EN COURS] Analyse de l'image : {nom_image}...")
        
        # Execution de l'inference via la fonction de la Cellule 2
        predictions = scan_image(image_path)
        
        print(f"[TERMINE] {len(predictions)} objet(s) detecte(s).")
        
        # Affichage et sauvegarde
        titre = f"Rapport d'Inventaire - {nom_image}"
        display_and_save_strategic_map(image_path, predictions, title=titre, output_dir=OUTPUT_DIR)
        
    print("-" * 60)
    print(f"[FIN DE LA MISSION] Toutes les images ont ete traitees et sauvegardees dans le dossier '{OUTPUT_DIR}'.")

In [ ]:
# ==========================================
# CELLULE 4 : BRIQUE 2 - ALERTE SINISTRE (SYSTEME DE PONDERATION)   [V2 : couleurs, étiquettes, légende T0 -> T1]
# ==========================================
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from collections import Counter

# 1. Dossier de sortie
OUTPUT_DIR_SINISTRE = "Alerte_Sinistre_Resultats"
os.makedirs(OUTPUT_DIR_SINISTRE, exist_ok=True)
print(f"[DOSSIER CREE] Rapports sauvegardes dans : {OUTPUT_DIR_SINISTRE}")

# 2. Matrice de ponderation (Risk Weights) pour les 22 classes
CLASS_WEIGHTS = {
    # Infrastructures statiques (Critique = 10.0)
    'harbor': 10.0, 'bridge': 10.0, 'building': 10.0, 'storage-tank': 10.0,
    'tennis-court': 10.0, 'swimming-pool': 10.0, 'basketball-court': 10.0, 
    'soccer-ball-field': 10.0, 'ground-track-field': 10.0, 'baseball-diamond': 10.0, 
    'roundabout': 10.0, 'construction-site': 10.0, 'infrastructure-diverse': 10.0,
    # Semi-statiques / Zones de stockage (Moyen = 1.0)
    'shipping-container': 1.0, 'vehicle-lot': 1.0,
    # Mobiles (Faible = 0.1)
    'plane': 0.1, 'helicopter': 0.1, 'ship': 0.1, 'small-vehicle': 0.1, 
    'large-vehicle': 0.1, 'railway-vehicle': 0.1, 'engineering-vehicle': 0.1
}

ALERT_THRESHOLD = 10.0

# 3. Moteur d'affichage avec 3 panneaux
def display_smart_disaster_report(path_t0, preds_t0, path_t1, preds_t1, pertes, total_score, output_dir):
    img_t0 = np.array(Image.open(path_t0).convert("RGB"))
    img_t1 = np.array(Image.open(path_t1).convert("RGB"))
    
    # [V2] Couleurs pour TOUTES les classes (T0 et T1).
    # En V1, une classe absente de T0 recevait le bleu de secours, confondu avec le bleu de « bridge ».
    counts_t0 = Counter(pred.category.name for pred in preds_t0)
    counts_t1 = Counter(pred.category.name for pred in preds_t1)
    toutes_classes = sorted(set(counts_t0) | set(counts_t1))
    class_colors = {cls: COLORS[i % len(COLORS)] for i, cls in enumerate(toutes_classes)}
    
    # Disposition : 3 colonnes (T0, T1, Panneau de donnees)
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 10), gridspec_kw={'width_ratios': [1, 1, 0.6]})
    fig.patch.set_facecolor('#f8f9fa')
    
    # --- Panneau 1 : T0 ---
    ax1.imshow(img_t0)
    ax1.set_title(f"Reference (T0)\n{os.path.basename(path_t0)}", fontsize=14, pad=10)
    ax1.axis("off")
    for pred in preds_t0:
        bbox = pred.bbox
        cls_name = pred.category.name
        color = class_colors.get(cls_name, 'blue')
        rect = mpatches.Rectangle((bbox.minx, bbox.miny), bbox.maxx - bbox.minx, bbox.maxy - bbox.miny,
                                  linewidth=2, edgecolor=color, facecolor='none', alpha=0.8)
        ax1.add_patch(rect)
        ax1.text(bbox.minx, bbox.miny - 4, cls_name, color=color, fontsize=7, fontweight='bold')   # [V2] étiquette

    # --- Panneau 2 : T1 ---
    ax2.imshow(img_t1)
    ax2.set_title(f"Post-Evenement (T1)\n{os.path.basename(path_t1)}", fontsize=14, pad=10)
    ax2.axis("off")
    for pred in preds_t1:
        bbox = pred.bbox
        cls_name = pred.category.name
        color = class_colors.get(cls_name, 'blue')
        rect = mpatches.Rectangle((bbox.minx, bbox.miny), bbox.maxx - bbox.minx, bbox.maxy - bbox.miny,
                                  linewidth=2, edgecolor=color, facecolor='none', alpha=0.8)
        ax2.add_patch(rect)
        ax2.text(bbox.minx, bbox.miny - 4, cls_name, color=color, fontsize=7, fontweight='bold')   # [V2] étiquette
        
    # --- Panneau 3 : Rapport et Legende ---
    ax3.axis("off")
    y_pos = 0.95
    
    # Titre d'alerte selon le score
    if total_score >= ALERT_THRESHOLD:
        titre_alerte = "ALERTE CRITIQUE\nDESTRUCTION DETECTEE"
        couleur_alerte = 'darkred'
    elif total_score > 0:
        titre_alerte = "MOUVEMENT NORMAL\nAUCUNE DESTRUCTION MAJEURE"
        couleur_alerte = 'darkorange'
    else:
        titre_alerte = "STATUT NORMAL\nINFRASTRUCTURES INTACTES"
        couleur_alerte = 'green'

    ax3.text(0.5, y_pos, titre_alerte, fontsize=18, fontweight='bold', color=couleur_alerte, ha='center', va='top')
    
    # Affichage du score
    y_pos -= 0.15
    ax3.text(0.5, y_pos, f"Score de Risque : {total_score:.1f} / {ALERT_THRESHOLD}", 
             fontsize=16, ha='center', va='top', bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray'))
    
    # Detail des pertes
    y_pos -= 0.12
    if pertes:
        # Correction de l'erreur ici : suppression de underline=True
        ax3.text(0.5, y_pos, "Details des disparitions :", fontsize=14, fontweight='bold', ha='center', va='top')
        y_pos -= 0.05
        for cls, count in pertes.items():
            poids_unitaire = CLASS_WEIGHTS.get(cls, 1.0)
            impact = count * poids_unitaire
            detail_txt = f"- {cls} : {count} perdu(s)\n  (Impact : +{impact:.1f} pts)"
            ax3.text(0.1, y_pos, detail_txt, fontsize=12, ha='left', va='top')
            y_pos -= 0.08
    
    # Legende des couleurs
    y_pos -= 0.05
    # [V2] Légende T0 -> T1 : on lit directement « bridge : 2 -> 0 », « harbor : 0 -> 5 »
    ax3.text(0.5, y_pos, "Inventaire T0 -> T1 :", fontsize=14, fontweight='bold', ha='center', va='top')
    y_pos -= 0.05
    legend_patches = [mpatches.Patch(color=class_colors[c], label=f"{c} : {counts_t0.get(c, 0)} -> {counts_t1.get(c, 0)}")
                      for c in toutes_classes]
    ax3.legend(handles=legend_patches, loc='upper center', bbox_to_anchor=(0.5, y_pos), fontsize='12', frameon=False)

    plt.tight_layout()
    
    # Sauvegarde
    nom_fichier = f"Rapport_Pondere_{os.path.basename(path_t0)}_vs_{os.path.basename(path_t1)}.png"
    chemin_sauvegarde = os.path.join(output_dir, nom_fichier)
    plt.savefig(chemin_sauvegarde, dpi=300, bbox_inches='tight')
    print(f"[SAUVEGARDE] Rapport enregistre sous : {chemin_sauvegarde}")
    
    plt.show()

# 4. Lancement de l'analyse
print("\n[ANALYSE TEMPORELLE PONDEREE - EVALUATION DES RISQUES]")
path_t0 = input("Entrez le chemin absolu de l'image initiale (T0) : ").strip()
path_t1 = input("Entrez le chemin absolu de l'image post-evenement (T1) : ").strip()

if not os.path.isfile(path_t0) or not os.path.isfile(path_t1):
    print("[ERREUR] L'un des chemins specifies est introuvable.")
else:
    print("\n[SCAN] Analyse T0...")
    preds_t0 = scan_image(path_t0)
    print("[SCAN] Analyse T1...")
    preds_t1 = scan_image(path_t1)
    
    counts_t0 = Counter([pred.category.name for pred in preds_t0])
    counts_t1 = Counter([pred.category.name for pred in preds_t1])
    
    # Calcul des pertes et du score de risque
    objets_disparus = {}
    score_risque_total = 0.0
    
    for cls, count_initial in counts_t0.items():
        count_final = counts_t1.get(cls, 0)
        if count_initial > count_final:
            perte = count_initial - count_final
            objets_disparus[cls] = perte
            
            # Application du systeme de poids
            poids = CLASS_WEIGHTS.get(cls, 1.0)
            score_risque_total += (perte * poids)
            
    print("-" * 60)
    print(f"[BILAN] Score de Risque Calcule : {score_risque_total:.1f}")
    if score_risque_total >= ALERT_THRESHOLD:
        print("[ALERTE ROUGE] Le score depasse le seuil critique (10.0). Destruction ou mouvement de masse suspect.")
    else:
        print("[RAS] Le score est sous le seuil critique. Variations considerees comme normales.")
    print("-" * 60)
    
    display_smart_disaster_report(path_t0, preds_t0, path_t1, preds_t1, objets_disparus, score_risque_total, OUTPUT_DIR_SINISTRE)

## Cellule 4 bis · Analyse des transformations (nouveauté V2)

La cellule 4 compare des **comptages par classe** (« 2 ponts à T0, 0 à T1 »). Elle ne sait pas **où** les choses ont changé, et elle ne voit pas qu'un `bridge` est devenu un `harbor` au même endroit.

Cette cellule **apparie chaque objet de T0 avec ce qui le recouvre à T1** et rend un verdict par objet. Le recouvrement mesure la part du plus petit des deux objets qui est couverte par l'autre : un petit débris posé sur l'emplacement d'un grand barrage compte bien comme « au même endroit ».

| Verdict | Condition | Compte dans le score ? |
|---|---|---|
| **INCHANGÉ** | même classe au même endroit | non |
| **DISPARU** | plus rien à cet endroit | **oui** |
| **TRANSFORMATION PLAUSIBLE** | autre classe, et la transition a un sens physique (`bridge` → `harbor` = débris) | **oui, à confirmer** |
| **INCOHÉRENCE MODÈLE** | autre classe, mais la transition n'a pas de sens (`tennis-court` → `ship`) | non : hésitation probable du modèle |

**Pourquoi « à confirmer » ?** Un changement d'étiquette peut être un vrai changement sur le terrain, **ou** une simple hésitation du modèle (lumière, saison, résolution). L'image seule ne tranche pas. Il faut une **source indépendante** : par exemple l'alerte texte de **Spirit Digger** (`CONFIRMATION_EXTERNE` ci-dessous).

> Cette cellule réutilise `path_t0`, `path_t1`, `preds_t0` et `preds_t1` calculés par la cellule 4 : **exécute la cellule 4 d'abord**. Aucun nouveau scan n'est lancé.


In [ ]:
# ==========================================
# CELLULE 4 bis : ANALYSE DES TRANSFORMATIONS (APPARIEMENT SPATIAL T0 -> T1)   [V2]
# ==========================================
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from PIL import Image
from collections import Counter

OUTPUT_DIR_TRANSFO = "Alerte_Transformation_Resultats"
os.makedirs(OUTPUT_DIR_TRANSFO, exist_ok=True)

# Transitions qui ont un sens physique après un événement (destruction -> débris, démolition -> chantier...)
TRANSITIONS_PLAUSIBLES = {
    "bridge": {"harbor", "infrastructure-diverse", "construction-site"},
    "building": {"construction-site", "infrastructure-diverse"},
    "storage-tank": {"construction-site", "infrastructure-diverse"},
    "harbor": {"infrastructure-diverse", "construction-site"},
    "infrastructure-diverse": {"construction-site"},
    "soccer-ball-field": {"construction-site"},
    "ground-track-field": {"construction-site"},
}
RECOUVREMENT_MIN = 0.30   # part minimale du plus petit objet recouverte pour dire « même endroit »

# Confirmation par une source indépendante (ex. l'alerte texte de Spirit Digger). Laisser à None si aucune.
CONFIRMATION_EXTERNE = None
# CONFIRMATION_EXTERNE = {"source": "Spirit Digger", "reference": "SD-20230606-Kakhovka", "cotation": "B1"}

COULEUR_VERDICT = {"DISPARU": "#d62728", "TRANSFORMATION PLAUSIBLE": "#ff7f0e",
                   "INCOHÉRENCE MODÈLE": "#7f7f7f", "INCHANGÉ": "#2ca02c"}
COMPTE_DANS_SCORE = {"DISPARU", "TRANSFORMATION PLAUSIBLE"}


def recouvrement(a, b):
    """Part du PLUS PETIT des deux objets qui est recouverte par l'autre (0 = disjoints, 1 = inclus).

    On n'utilise pas l'IoU classique : un petit débris entièrement situé sur l'ancien barrage aurait
    un IoU très faible (car le barrage est immense), alors qu'il est bien « au même endroit ».
    """
    x1, y1 = max(a.minx, b.minx), max(a.miny, b.miny)
    x2, y2 = min(a.maxx, b.maxx), min(a.maxy, b.maxy)
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    aire = lambda r: (r.maxx - r.minx) * (r.maxy - r.miny)
    return inter / (min(aire(a), aire(b)) + 1e-9)


def analyser_transformations(preds_t0, preds_t1):
    """Un verdict par objet de T0, et la liste des objets de T1 qui le recouvrent."""
    resultats = []
    for p0 in preds_t0:
        c0 = p0.category.name
        voisins = [(recouvrement(p0.bbox, p1.bbox), p1) for p1 in preds_t1]
        voisins = [(v, p1) for v, p1 in voisins if v >= RECOUVREMENT_MIN]
        classes_t1 = {p1.category.name for _, p1 in voisins}

        if not voisins:
            verdict = "DISPARU"
        elif c0 in classes_t1:
            verdict = "INCHANGÉ"
        elif classes_t1 & TRANSITIONS_PLAUSIBLES.get(c0, set()):
            verdict = "TRANSFORMATION PLAUSIBLE"
        else:
            verdict = "INCOHÉRENCE MODÈLE"

        resultats.append({
            "pred_t0": p0,
            "voisins_t1": [p1 for _, p1 in voisins],
            "classe T0": c0,
            "confiance T0": round(p0.score.value, 2),
            "verdict": verdict,
            "devenu (T1)": " + ".join(sorted(classes_t1)) or "-",
            "fragments T1": len(voisins),   # un grand objet découpé en plusieurs morceaux = indice de destruction
            "recouvrement max": round(max((v for v, _ in voisins), default=0.0), 2),
            "impact": CLASS_WEIGHTS.get(c0, 1.0) if verdict in COMPTE_DANS_SCORE else 0.0,
        })
    return resultats


def display_transformation_report(path_t0, path_t1, resultats, preds_t1, score, output_dir):
    img_t0 = np.array(Image.open(path_t0).convert("RGB"))
    img_t1 = np.array(Image.open(path_t1).convert("RGB"))
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 10), gridspec_kw={'width_ratios': [1, 1, 0.6]})
    fig.patch.set_facecolor('#f8f9fa')

    # --- T0 : chaque objet coloré selon son VERDICT ---
    ax1.imshow(img_t0)
    ax1.set_title(f"Reference (T0) - couleur = verdict\n{os.path.basename(path_t0)}", fontsize=14, pad=10)
    ax1.axis("off")
    for r in resultats:
        b, couleur = r["pred_t0"].bbox, COULEUR_VERDICT[r["verdict"]]
        ax1.add_patch(mpatches.Rectangle((b.minx, b.miny), b.maxx - b.minx, b.maxy - b.miny,
                                         linewidth=2.5, edgecolor=couleur, facecolor='none'))
        ax1.text(b.minx, b.miny - 4, f"{r['classe T0']} : {r['verdict']}", color=couleur, fontsize=7, fontweight='bold')

    # --- T1 : en orange pointillé, ce qu'est devenu un objet transformé ; le reste en gris fin ---
    transformes = {id(p1) for r in resultats if r["verdict"] == "TRANSFORMATION PLAUSIBLE" for p1 in r["voisins_t1"]}
    ax2.imshow(img_t1)
    ax2.set_title(f"Post-Evenement (T1)\n{os.path.basename(path_t1)}", fontsize=14, pad=10)
    ax2.axis("off")
    for p1 in preds_t1:
        b = p1.bbox
        if id(p1) in transformes:
            couleur, style, epaisseur = COULEUR_VERDICT["TRANSFORMATION PLAUSIBLE"], '--', 2.5
        else:
            couleur, style, epaisseur = '#9a9a9a', '-', 1
        ax2.add_patch(mpatches.Rectangle((b.minx, b.miny), b.maxx - b.minx, b.maxy - b.miny,
                                         linewidth=epaisseur, edgecolor=couleur, facecolor='none', linestyle=style))
        ax2.text(b.minx, b.miny - 4, p1.category.name, color=couleur, fontsize=7, fontweight='bold')

    # --- Panneau de synthèse ---
    ax3.axis("off")
    n_transfo = sum(r["verdict"] == "TRANSFORMATION PLAUSIBLE" for r in resultats)
    if score >= ALERT_THRESHOLD and CONFIRMATION_EXTERNE:
        titre, couleur = "ALERTE MAXIMALE\nDESTRUCTION CONFIRMEE\n(image + source independante)", 'darkred'
    elif score >= ALERT_THRESHOLD and n_transfo:
        titre, couleur = "ALERTE\nTRANSFORMATION A CONFIRMER", '#c85a00'
    elif score >= ALERT_THRESHOLD:
        titre, couleur = "ALERTE CRITIQUE\nDESTRUCTION DETECTEE", 'darkred'
    else:
        titre, couleur = "STATUT NORMAL\nPAS DE CHANGEMENT MAJEUR", 'green'
    ax3.text(0.5, 0.97, titre, fontsize=17, fontweight='bold', color=couleur, ha='center', va='top')
    ax3.text(0.5, 0.78, f"Score de risque spatial : {score:.1f} / {ALERT_THRESHOLD}", fontsize=15, ha='center',
             va='top', bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray'))

    y = 0.68
    ax3.text(0.5, y, "Verdicts (objets de T0) :", fontsize=13, fontweight='bold', ha='center', va='top')
    for verdict, n in Counter(r["verdict"] for r in resultats).most_common():
        y -= 0.055
        ax3.text(0.12, y, f"■ {verdict} : {n}", color=COULEUR_VERDICT[verdict], fontsize=12, fontweight='bold', va='top')

    y -= 0.09
    if CONFIRMATION_EXTERNE:
        msg = (f"Confirmation : {CONFIRMATION_EXTERNE['source']}\n"
               f"ref. {CONFIRMATION_EXTERNE['reference']} (cotation {CONFIRMATION_EXTERNE['cotation']})")
    elif n_transfo:
        msg = "Changement d'etiquette = vrai changement\nOU hesitation du modele.\nConfirmation requise par une\nsource independante (ex. Spirit Digger)."
    else:
        msg = ""
    ax3.text(0.5, y, msg, fontsize=11, ha='center', va='top', style='italic')

    plt.tight_layout()
    nom = f"Rapport_Transformations_{os.path.basename(path_t0)}_vs_{os.path.basename(path_t1)}.png"
    chemin = os.path.join(output_dir, nom)
    plt.savefig(chemin, dpi=300, bbox_inches='tight')
    print(f"[SAUVEGARDE] Rapport enregistre sous : {chemin}")
    plt.show()


# Lancement (réutilise les scans de la cellule 4)
resultats = analyser_transformations(preds_t0, preds_t1)
score_spatial = sum(r["impact"] for r in resultats)

tableau = pd.DataFrame([{k: v for k, v in r.items() if k not in ("pred_t0", "voisins_t1")} for r in resultats])
print("-" * 60)
print(f"[BILAN SPATIAL] Score : {score_spatial:.1f} (score par comptage de la cellule 4 : {score_risque_total:.1f})")
print("-" * 60)
display(tableau)
display_transformation_report(path_t0, path_t1, resultats, preds_t1, score_spatial, OUTPUT_DIR_TRANSFO)


In [ ]:
# ==========================================
# CELLULE 5 : BRIQUE 3 - ALERTE INTRUSION (DETECTION D'APPARITIONS)
# ==========================================
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from collections import Counter

# 1. Dossier de sortie pour les intrusions
OUTPUT_DIR_INTRUSION = "Alerte_Intrusion_Resultats"
os.makedirs(OUTPUT_DIR_INTRUSION, exist_ok=True)
print(f"[DOSSIER CREE] Rapports d'intrusion sauvegardes dans : {OUTPUT_DIR_INTRUSION}")

# 2. Matrice de ponderation (Risk Weights) specifique aux intrusions
INTRUSION_WEIGHTS = {
    # Mobiles (Menace Critique = 10.0) : Un vehicule ou un avion dans une zone interdite est une alerte majeure
    'plane': 10.0, 'helicopter': 10.0, 'ship': 10.0, 'small-vehicle': 10.0, 
    'large-vehicle': 10.0, 'railway-vehicle': 10.0, 'engineering-vehicle': 10.0,
    # Semi-statiques / Deploiement (Moyen = 5.0)
    'shipping-container': 5.0, 'vehicle-lot': 5.0, 'construction-site': 5.0,
    # Infrastructures (Faible/Moyen = 2.0) : Rarement une apparition soudaine, mais indique une construction illegale
    'building': 2.0, 'storage-tank': 2.0, 'infrastructure-diverse': 2.0
}

INTRUSION_THRESHOLD = 10.0

# 3. Moteur d'affichage avec 3 panneaux oriente "Intrusion"
def display_intrusion_report(path_t0, preds_t0, path_t1, preds_t1, apparitions, total_score, output_dir):
    img_t0 = np.array(Image.open(path_t0).convert("RGB"))
    img_t1 = np.array(Image.open(path_t1).convert("RGB"))
    
    # Creation des couleurs pour les classes detectees a T1 (puisque c'est la qu'il y a des nouveautes)
    labels_t1 = [pred.category.name for pred in preds_t1]
    unique_classes = list(set(labels_t1))
    class_colors = {cls: COLORS[i % len(COLORS)] for i, cls in enumerate(unique_classes)}
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 10), gridspec_kw={'width_ratios': [1, 1, 0.6]})
    fig.patch.set_facecolor('#f4f6f9') # Bleu-gris tres leger
    
    # --- Panneau 1 : T0 ---
    ax1.imshow(img_t0)
    ax1.set_title(f"Reference Zone (T0)\n{os.path.basename(path_t0)}", fontsize=14, pad=10)
    ax1.axis("off")
    for pred in preds_t0:
        bbox = pred.bbox
        cls_name = pred.category.name
        color = class_colors.get(cls_name, 'blue')
        rect = mpatches.Rectangle((bbox.minx, bbox.miny), bbox.maxx - bbox.minx, bbox.maxy - bbox.miny,
                                  linewidth=2, edgecolor=color, facecolor='none', alpha=0.5, linestyle='--')
        ax1.add_patch(rect)

    # --- Panneau 2 : T1 ---
    ax2.imshow(img_t1)
    ax2.set_title(f"Scan Actuel (T1)\n{os.path.basename(path_t1)}", fontsize=14, pad=10)
    ax2.axis("off")
    for pred in preds_t1:
        bbox = pred.bbox
        cls_name = pred.category.name
        color = class_colors.get(cls_name, 'blue')
        # Mise en evidence si c'est une classe qui a vu une apparition
        linewidth = 3 if cls_name in apparitions else 1
        alpha = 1.0 if cls_name in apparitions else 0.4
        
        rect = mpatches.Rectangle((bbox.minx, bbox.miny), bbox.maxx - bbox.minx, bbox.maxy - bbox.miny,
                                  linewidth=linewidth, edgecolor=color, facecolor='none', alpha=alpha)
        ax2.add_patch(rect)
        
    # --- Panneau 3 : Rapport et Legende ---
    ax3.axis("off")
    y_pos = 0.95
    
    if total_score >= INTRUSION_THRESHOLD:
        titre_alerte = "VIOLATION DE ZONE\nINTRUSION DETECTEE"
        couleur_alerte = 'darkred'
    elif total_score > 0:
        titre_alerte = "ACTIVITE MINEURE\nNOUVEAUX ELEMENTS PRESENTS"
        couleur_alerte = 'darkorange'
    else:
        titre_alerte = "ZONE SECURISEE\nAUCUNE INTRUSION"
        couleur_alerte = 'green'

    ax3.text(0.5, y_pos, titre_alerte, fontsize=18, fontweight='bold', color=couleur_alerte, ha='center', va='top')
    
    y_pos -= 0.15
    ax3.text(0.5, y_pos, f"Score de Menace : {total_score:.1f} / {INTRUSION_THRESHOLD}", 
             fontsize=16, ha='center', va='top', bbox=dict(facecolor='white', alpha=0.9, edgecolor='black'))
    
    y_pos -= 0.12
    if apparitions:
        ax3.text(0.5, y_pos, "Details des apparitions (Intrus) :", fontsize=14, fontweight='bold', ha='center', va='top')
        y_pos -= 0.05
        for cls, count in apparitions.items():
            poids_unitaire = INTRUSION_WEIGHTS.get(cls, 1.0)
            impact = count * poids_unitaire
            detail_txt = f"+ {count} {cls} detecte(s)\n  (Niveau de menace : +{impact:.1f} pts)"
            ax3.text(0.1, y_pos, detail_txt, fontsize=12, ha='left', va='top', color='darkred', fontweight='bold')
            y_pos -= 0.08
    
    y_pos -= 0.05
    ax3.text(0.5, y_pos, "Legende des objets detectes a T1 :", fontsize=14, fontweight='bold', ha='center', va='top')
    y_pos -= 0.05
    legend_patches = [mpatches.Patch(color=color, label=cls) for cls, color in class_colors.items()]
    if legend_patches:
        ax3.legend(handles=legend_patches, loc='upper center', bbox_to_anchor=(0.5, y_pos), fontsize='12', frameon=False)

    plt.tight_layout()
    
    nom_fichier = f"Rapport_Intrusion_{os.path.basename(path_t0)}_vs_{os.path.basename(path_t1)}.png"
    chemin_sauvegarde = os.path.join(output_dir, nom_fichier)
    plt.savefig(chemin_sauvegarde, dpi=300, bbox_inches='tight')
    print(f"[SAUVEGARDE] Rapport d'intrusion enregistre sous : {chemin_sauvegarde}")
    
    plt.show()

# 4. Lancement de l'analyse
print("\n[CONTROLE D'ACCES ET DETECTION D'INTRUSION]")
path_t0 = input("Entrez le chemin absolu de l'image de la zone vide/securisee (T0) : ").strip()
path_t1 = input("Entrez le chemin absolu de l'image suspecte (T1) : ").strip()

if not os.path.isfile(path_t0) or not os.path.isfile(path_t1):
    print("[ERREUR] L'un des chemins specifies est introuvable.")
else:
    print("\n[SCAN] Analyse de la zone de reference T0...")
    preds_t0 = scan_image(path_t0)
    print("[SCAN] Analyse de la zone actuelle T1...")
    preds_t1 = scan_image(path_t1)
    
    counts_t0 = Counter([pred.category.name for pred in preds_t0])
    counts_t1 = Counter([pred.category.name for pred in preds_t1])
    
    # Recherche des apparitions (presentes a T1 mais absentes ou en moindre quantite a T0)
    objets_apparus = {}
    score_menace_total = 0.0
    
    for cls, count_final in counts_t1.items():
        count_initial = counts_t0.get(cls, 0)
        if count_final > count_initial:
            apparition = count_final - count_initial
            objets_apparus[cls] = apparition
            
            poids = INTRUSION_WEIGHTS.get(cls, 1.0)
            score_menace_total += (apparition * poids)
            
    print("-" * 60)
    print(f"[BILAN] Score de Menace Calcule : {score_menace_total:.1f}")
    if score_menace_total >= INTRUSION_THRESHOLD:
        print("[ALERTE CRITIQUE] Intrusion detectee. Le score depasse le seuil tolere (10.0).")
    elif score_menace_total > 0:
        print("[AVERTISSEMENT] Activite anormale detectee, mais sous le seuil critique.")
    else:
        print("[RAS] Aucun objet intrus detecte. La zone est securisee.")
    print("-" * 60)
    
    display_intrusion_report(path_t0, preds_t0, path_t1, preds_t1, objets_apparus, score_menace_total, OUTPUT_DIR_INTRUSION)

In [ ]:
# ==========================================
# CELLULE 6 : BRIQUE 4 - CORRELATION SPATIALE & DASHBOARD FINAL
# ==========================================
import os
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from PIL import Image
from collections import Counter

# 1. Configuration
OUTPUT_DIR_ATTAQUE = "Alerte_Attaque_Resultats"
os.makedirs(OUTPUT_DIR_ATTAQUE, exist_ok=True)
print(f"[DOSSIER CREE] Rapports d'investigation sauvegardes dans : {OUTPUT_DIR_ATTAQUE}")

MOBILE_THREATS = [
    'ship', 'plane', 'helicopter', 'small-vehicle', 'large-vehicle'
]

TOLERANCE_SPATIALE = 60.0 

def get_centroid(bbox):
    return ((bbox.minx + bbox.maxx) / 2, (bbox.miny + bbox.maxy) / 2)

def distance_points(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

# 2. Moteur d'affichage - Dashboard Avance
def display_dashboard_report(paths, all_preds, suspects, destructions, output_dir):
    fig = plt.figure(figsize=(24, 14))
    fig.patch.set_facecolor('#ffffff')
    
    gs = gridspec.GridSpec(2, 3, height_ratios=[1.3, 1], wspace=0.1, hspace=0.15)
    
    # Titre simplifie
    fig.suptitle("ANALYSE DE CHRONOLOGIE COMPLEXE", 
                 fontsize=22, fontweight='bold', color='#1a1a1a', y=0.98)
    
    titles = [
        f"T0 : Situation Initiale\n{os.path.basename(paths[0])}",
        f"T1 : Detection d'Intrusion\n{os.path.basename(paths[1])}",
        f"T2 : Bilan Post-Evenement\n{os.path.basename(paths[2])}"
    ]
    
    # --- LIGNE 1 : Affichage des images chronologiques ---
    images_np = [np.array(Image.open(p).convert("RGB")) for p in paths]
    
    for i in range(3):
        ax = fig.add_subplot(gs[0, i])
        ax.imshow(images_np[i])
        ax.set_title(titles[i], fontsize=12, color='black', pad=8, fontweight='bold')
        ax.axis("off")
        
        preds = all_preds[i]
        
        for pred in preds:
            if pred in suspects and i == 1:
                continue 
            
            bbox = pred.bbox
            rect = mpatches.Rectangle((bbox.minx, bbox.miny), bbox.maxx - bbox.minx, bbox.maxy - bbox.miny,
                                      linewidth=1, edgecolor='cyan', facecolor='none', alpha=0.3)
            ax.add_patch(rect)
            
        # Surbrillance des Suspects (T1)
        if i == 1:
            for suspect in suspects:
                bbox = suspect.bbox
                rect = mpatches.Rectangle((bbox.minx, bbox.miny), bbox.maxx - bbox.minx, bbox.maxy - bbox.miny,
                                          linewidth=3, edgecolor='red', facecolor='none', alpha=1.0)
                ax.add_patch(rect)
                cx, cy = get_centroid(bbox)
                ax.plot(cx, cy, marker='+', color='red', markersize=12, markeredgewidth=2)

        # Surbrillance des Destructions (T2)
        if i == 2:
            for detruit in destructions:
                bbox = detruit.bbox
                rect = mpatches.Rectangle((bbox.minx, bbox.miny), bbox.maxx - bbox.minx, bbox.maxy - bbox.miny,
                                          linewidth=2, edgecolor='darkorange', facecolor='none', alpha=1.0, linestyle='--')
                ax.add_patch(rect)

    # --- LIGNE 2 - GAUCHE : Zoom sur le suspect ---
    ax_zoom = fig.add_subplot(gs[1, 0])
    if suspects:
        s = suspects[0]
        bbox = s.bbox
        cx, cy = int(get_centroid(bbox)[0]), int(get_centroid(bbox)[1])
        
        pad = 150 
        img_t1 = images_np[1]
        y1, y2 = max(0, cy - pad), min(img_t1.shape[0], cy + pad)
        x1, x2 = max(0, cx - pad), min(img_t1.shape[1], cx + pad)
        
        zoom_img = img_t1[y1:y2, x1:x2]
        ax_zoom.imshow(zoom_img)
        
        zx = bbox.minx - x1
        zy = bbox.miny - y1
        zw = bbox.maxx - bbox.minx
        zh = bbox.maxy - bbox.miny
        
        rect = mpatches.Rectangle((zx, zy), zw, zh, linewidth=4, edgecolor='red', facecolor='none')
        ax_zoom.add_patch(rect)
        ax_zoom.set_title("FOCUS CAMERA : SUSPECT ISOLE", fontsize=14, fontweight='bold', color='darkred')
        ax_zoom.axis("off")
    else:
        ax_zoom.text(0.5, 0.5, "AUCUN SUSPECT\nA CIBLER", ha='center', va='center', fontsize=16, color='gray')
        ax_zoom.axis("off")

    # --- LIGNE 2 - DROITE : Rapport de synthese ---
    ax_report = fig.add_subplot(gs[1, 1:])
    ax_report.axis("off")
    
    y_pos = 0.95
    ax_report.text(0.5, y_pos, "SYNTHESE DE L'INVESTIGATION", fontsize=18, fontweight='bold', color='black', ha='center', va='top')
    
    # Phase 1 avec agregation et sans coordonnees
    y_pos -= 0.15
    ax_report.text(0.05, y_pos, "[PHASE 1] T0 -> T1 : TRACKING DES INTRUSIONS", fontsize=14, fontweight='bold', color='darkred', ha='left')
    y_pos -= 0.10
    if suspects:
        suspect_counts = Counter([s.category.name for s in suspects])
        for cls, count in suspect_counts.items():
            ax_report.text(0.1, y_pos, f"-> Apparition de {count} x '{cls}'", fontsize=12, color='black', ha='left')
            y_pos -= 0.08
    else:
        ax_report.text(0.1, y_pos, "-> Aucun nouvel intrus detecte par rapport a T0.", fontsize=12, color='dimgray', ha='left')
        y_pos -= 0.08
        
    # Phase 2 avec agregation
    y_pos -= 0.05
    ax_report.text(0.05, y_pos, "[PHASE 2] T1 -> T2 : LOCALISATION DES DEGATS", fontsize=14, fontweight='bold', color='darkorange', ha='left')
    y_pos -= 0.10
    if destructions:
        dest_counts = Counter([d.category.name for d in destructions])
        for cls, count in dest_counts.items():
            ax_report.text(0.1, y_pos, f"-> Destruction de {count} x '{cls}' confirmee", fontsize=12, color='black', ha='left')
            y_pos -= 0.08
    else:
        ax_report.text(0.1, y_pos, "-> Aucune perte spatiale detectee.", fontsize=12, color='dimgray', ha='left')
        y_pos -= 0.08
        
    y_pos -= 0.05
    legend_elements = [
        mpatches.Patch(edgecolor='cyan', facecolor='none', linewidth=1, label='Element statique / connu'),
        mpatches.Patch(edgecolor='red', facecolor='none', linewidth=3, label='Suspect Isole (Nouvelle apparition)'),
        mpatches.Patch(edgecolor='darkorange', facecolor='none', linewidth=2, linestyle='--', label='Zone de destruction confirmee')
    ]
    ax_report.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(0.05, y_pos), fontsize=10, frameon=True)
    
    y_pos -= 0.35
    if suspects and destructions:
        verdict = "ALERTE MAXIMALE : ATTAQUE CONFIRMEE\nLien etabli entre l'intrus isole et les destructions localisees."
        color_verdict = 'darkred'
    elif destructions:
        verdict = "ALERTE : DESTRUCTION ISOLEE\nPerte d'element sans cible mobile suspecte identifiee."
        color_verdict = 'darkorange'
    elif suspects:
        verdict = "SURVEILLANCE : INTRUS DETECTE\nPresence d'une cible non autorisee, aucun degat pour le moment."
        color_verdict = 'goldenrod'
    else:
        verdict = "STATUT : NORMAL\nAucune anomalie spatiale detectee sur la chronologie."
        color_verdict = 'green'
        
    ax_report.text(0.5, y_pos, verdict, fontsize=14, fontweight='bold', color=color_verdict, ha='center', va='top', 
                   bbox=dict(facecolor='#f8f9fa', alpha=1.0, edgecolor=color_verdict, boxstyle='round,pad=1', linewidth=2))

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    
    nom_fichier = f"Dashboard_Final_{os.path.basename(paths[0])}.png"
    chemin_sauvegarde = os.path.join(output_dir, nom_fichier)
    plt.savefig(chemin_sauvegarde, dpi=300, bbox_inches='tight')
    print(f"[SAUVEGARDE] Tableau de bord enregistre sous : {chemin_sauvegarde}")
    
    plt.show()

# 3. Lancement du tracking
print("\n[INITIATION DU PROTOCOLE D'INVESTIGATION SPATIALE]")
path_t0 = input("Entrez le chemin absolu de l'image (T0 - Etat normal) : ").strip()
path_t1 = input("Entrez le chemin absolu de l'image (T1 - Arrivee du suspect) : ").strip()
path_t2 = input("Entrez le chemin absolu de l'image (T2 - Post-evenement) : ").strip()

paths = [path_t0, path_t1, path_t2]

if not all(os.path.isfile(p) for p in paths):
    print("[ERREUR] Un ou plusieurs chemins specifier sont introuvables.")
else:
    all_preds = []
    for i, p in enumerate(paths):
        print(f"[SCAN] Analyse spatiale en cours (T{i})...")
        preds = scan_image(p)
        all_preds.append(preds)
        
    suspects_isoles = []
    for pred_t1 in all_preds[1]:
        if pred_t1.category.name in MOBILE_THREATS:
            c1 = get_centroid(pred_t1.bbox)
            est_nouveau = True
            
            for pred_t0 in all_preds[0]:
                if pred_t0.category.name == pred_t1.category.name:
                    c0 = get_centroid(pred_t0.bbox)
                    if distance_points(c1, c0) < TOLERANCE_SPATIALE:
                        est_nouveau = False
                        break
                        
            if est_nouveau:
                suspects_isoles.append(pred_t1)
                
    destructions_localisees = []
    # On verifie desormais TOUS les objets presents a T1 pour voir s'ils ont disparu a T2
    for pred_t1 in all_preds[1]:
        c1 = get_centroid(pred_t1.bbox)
        est_detruit = True
        
        for pred_t2 in all_preds[2]:
            if pred_t2.category.name == pred_t1.category.name:
                c2 = get_centroid(pred_t2.bbox)
                if distance_points(c1, c2) < TOLERANCE_SPATIALE:
                    est_detruit = False
                    break
                    
        if est_detruit:
            destructions_localisees.append(pred_t1)

    print("-" * 60)
    print("[TRACKING TERMINE] Generation du rapport d'investigation propre...")
    print("-" * 60)
    
    display_dashboard_report(paths, all_preds, suspects_isoles, destructions_localisees, OUTPUT_DIR_ATTAQUE)

In [ ]:
import shutil
from pathlib import Path

base = Path("/kaggle/working")

dossiers = [
    "Alerte_Attaque_Resultats",
    "Alerte_Intrusion_Resultats",
    "Alerte_Sinistre_Resultats",
    "Inventaire_Detection_Resultats",
]

for dossier in dossiers:
    chemin = base / dossier
    
    if chemin.exists():
        print(f"Compression de {dossier}...")
        shutil.make_archive(
            str(base / dossier),
            "zip",
            root_dir=base,
            base_dir=dossier
        )
        print(f"✅ {dossier}.zip créé")
    else:
        print(f"❌ Dossier introuvable : {chemin}")


In [ ]:
from IPython.display import FileLink, display

for dossier in dossiers:
    zip_path = base / f"{dossier}.zip"
    display(FileLink(str(zip_path)))


In [ ]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/Alerte_Sinistre_Resultats.zip"))
